In [2]:
# Core imports man
import sys
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy
import time

# Numerical libraries
from scipy.integrate import solve_ivp, odeint, cumulative_trapezoid
from scipy.interpolate import (
    UnivariateSpline, splrep, splev, CubicSpline,
    interp1d, PchipInterpolator, InterpolatedUnivariateSpline
)

import numdifftools as nd

# Random number generator (GSL)
import pygsl.rng

# custom InflationModels code to path the one below is for wkb approximation method
sys.path.append(
    '/Users/epmeador/Desktop/research/rwarthur/inflation_gravitywaves/inflation_code/InflationModels'
)

# Enable spectra
SPECTRUM = True

# Local modules from InflationModels
from MacroDefinitions import *
from calcpath import *
from int_de import *
if SPECTRUM:
    from spectrum_OG_nanoscale_nodiagnostics import *

# ========================
# GLOBAL SETTINGS
# ========================
NEQS = 8
SPECTRUM = True
SAVEPATHS = True

NMAX = 1.2
NMIN = 0.3

# if I want to expand the lam5 space for higgs values I can do a symmetric expansion around what I know works
LAM5_MIN = -3e-8
LAM5_MAX = 2.0e-8
LAM5_BASE = -8.92461e-9
NUM_LAM5_GRID = 398

lam5_set = np.linspace(LAM5_MIN, LAM5_MAX, num=NUM_LAM5_GRID)
lam5_set = np.sort(np.append(lam5_set, [LAM5_BASE, 0.0]))

print(f"Total models: {len(lam5_set)}")
print(f"Base λ₅={LAM5_BASE:.6e} included: {LAM5_BASE in lam5_set}")
print(f"Zero included: {0.0 in lam5_set}")

# This sets how many nontrivial models you are running
NUMPOINTS = 1

# This will give us the min and max number of e-folds we are looking for
NUMEFOLDSMAX = 65.0
NUMEFOLDSMIN = 57.0

# Here we are writing out our output files
BASE_OUTDIR = "/Users/epmeador/Desktop/research/rwarthur/inflation_gravitywaves/inflation_code/Slow-Roll Parameters Tests/higgs_potential_tests/neqs8"

# RNG initialization process
my_random = pygsl.rng.ranlxd2()
my_random.set(0)
np.random.seed(0)

# ========================
# Support Functions
# ========================
class Calc:
    def __init__(self):
        self.Y = np.zeros(NEQS, dtype=float, order='C')
        self.initY = np.zeros(NEQS, dtype=float, order='C')
        self.ret = ""
        self.npoints = 0
        self.Nefolds = 0.0


def pick_init_vals(lam5):
    init_vals = np.zeros(NEQS, dtype=float, order='C')
    init_vals[0] = 5.5
    init_vals[1] = 1.0
    init_vals[2] = 0.000209237
    init_vals[3] = -0.0342419
    init_vals[4] = 0.000278972
    init_vals[5] = -4.60971e-06
    init_vals[6] = 6.87065e-08
    init_vals[7] = lam5

    init_Nefolds = my_random.uniform() * (NUMEFOLDSMAX - NUMEFOLDSMIN) + NUMEFOLDSMIN
    return init_vals, init_Nefolds


def we_should_calc_spec(y):
    return (specindex(y) > NMIN and specindex(y) < NMAX)


def we_should_save_path(retval, save, pointcount, printevery):
    return (retval == "nontrivial") and (not save) and (pointcount % printevery == 0)


def save_path(y, N, kount, fname):
    with open(fname, "w") as outfile:
        for i in range(kount):
            for j in range(NEQS):
                outfile.write("%le " % y[j, i])
            outfile.write("%lf " % N[i])

            V = (3. / (8. * np.pi)) * y[1, i] * y[1, i] * (1. - y[2, i] / 3.)
            outfile.write("%le %le\n" % (V, (V * y[2, i]) / (3. - y[2, i])))


# ========================
# Main Loop
# ========================
def run_neqs8_models():
    summary_records = []

    for lam5 in lam5_set:
        print(f"\n=== Running λ5 = {lam5:.1e} ===")

        OUTDIR = f"{BASE_OUTDIR}/lam5_{lam5:.1e}"
        os.makedirs(OUTDIR, exist_ok=True)
        OUTFILE1_NAME = f"{OUTDIR}/test_nr_neqs{NEQS}.dat"
        OUTFILE2_NAME = f"{OUTDIR}/test_esigma_neqs{NEQS}.dat"

        try:
            outfile1 = open(OUTFILE1_NAME, "w")
            outfile2 = open(OUTFILE2_NAME, "w")
        except IOError as e:
            print("Could not open output files: ", e)
            sys.exit()

        if SPECTRUM:
            u_s = np.empty((2, knos))
            u_t = np.empty((2, knos))
            y_final = np.empty(NEQS + 1)
            spec_count = 0

        calc = Calc()
        iters = 0
        points = 0
        outcount = 0
        asymcount = 0
        nontrivcount = 0
        insuffcount = 0
        noconvcount = 0
        badncount = 0
        errcount = 0
        savedone = 0

        while nontrivcount < NUMPOINTS:
            iters += 1
            if iters > 200:
                break

            if iters % 100 == 0:
                print(f"  Iter {iters}, nontriv={nontrivcount}")

            yinit, calc.Nefolds = pick_init_vals(lam5)
            y = yinit.copy()

            path = np.array([[]])
            N = np.array([])

            t0 = time.perf_counter()
            calc.ret = calcpath(calc.Nefolds, y, path, N, calc)
            t1 = time.perf_counter()
            print(f"calcpath runtime: {t1 - t0:.4f} s")
            print(f"  -> {calc.ret}")

            print("\n=== INTEGRATOR & INITIAL CONDITIONS DIAGNOSTIC ===")
            print(f"Initial values (yinit): {yinit}")
            print(f"  φ0 = {yinit[0]:.6e}")
            print(f"  H0 = {yinit[1]:.6e}")
            print(f"  ε0 = {yinit[2]:.6e}")
            print(f"  σ0 = {yinit[3]:.6e}")
            print(f"  λ₂ = {yinit[4]:.6e}")
            if len(yinit) > 5:
                print(f"  λ₃ = {yinit[5]:.6e}")
            if len(yinit) > 6:
                print(f"  λ₄ = {yinit[6]:.6e}")
            if len(yinit) > 7:
                print(f"  λ₅ = {yinit[7]:.6e}")
            print(f"Initial Nefolds = {calc.Nefolds:.3f}")

            try:
                import pygsl.odeiv as odeiv
                s = odeiv.step_rk4(len(yinit), derivs1)
                c = odeiv.control_y_new(s, 1e-8, 1e-8)
                print(f"GSL integrator class: {s.__class__.__name__}")
                print(f"Expected tolerances: atol=1e-8, rtol=1e-8")
            except Exception as e:
                print("Could not check GSL integrator:", e)

            print("===============================================\n")

            if calc.ret == "asymptote":
                asymcount += 1
                if asymcount > 100:
                    print("Too many asymptotes, stopping")
                    break
                continue

            if calc.ret == "nontrivial":
                r = tsratio(y)
                ns = specindex(y)
                alpha_s = dspecindex(y)
                outfile1.write(f"{r:.10f} {ns:.10f} {alpha_s:.10f}\n")
                outfile1.flush()

                for i in range(NEQS):
                    outfile2.write("%le " % y[i])
                outfile2.write("%f\n" % calc.Nefolds)
                outfile2.flush()

                points += 1
                savedone = 0
                nontrivcount += 1

                if SPECTRUM and we_should_calc_spec(y):
                    print(f"  ns = {specindex(y):.3f}")
                    print(f"  -> Evaluating spectrum {spec_count}")

                    y_final[:NEQS] = path[:NEQS, 3]
                    y_final[NEQS] = N[3]

                    t0 = time.perf_counter()
                    spectrum_status = spectrum(
                        y_final, y, u_s, u_t, calc.Nefolds,
                        derivs1, scalarsys, tensorsys
                    )
                    t1 = time.perf_counter()
                    print(f"spectrum runtime: {t1 - t0:.4f} s")

                    if spectrum_status:
                        errcount += 1

                    spec_s_name = f"{OUTDIR}/spec_s{spec_count:03d}_neqs{NEQS}.dat"
                    spec_t_name = f"{OUTDIR}/spec_t{spec_count:03d}_neqs{NEQS}.dat"
                    np.savetxt(spec_s_name, u_s[:, :knos].T)
                    np.savetxt(spec_t_name, u_t[:, :knos].T)
                    spec_count += 1

                if SPECTRUM:
                    print(f"  -> Before normalization: y[1] = {y[1]:.6e}")
                    for j in range(calc.npoints):
                        path[0, j] = path[0, j] - path[0, calc.npoints - 1]
                        path[1, j] = path[1, j] * y[1]

                    print(f"  -> After normalization: max(path[1,:]) = {np.max(path[1,:]):.6e}")

                path_name = f"{OUTDIR}/path_neqs{NEQS}_lam5{lam5:.1e}_{outcount:03d}.dat"
                print(f"  -> Saving path {path_name}")
                save_path(path, N, calc.npoints, path_name)
                outcount += 1

                summary_records.append({
                    "lam5": lam5,
                    "r": r,
                    "n_s": ns,
                    "alpha_s": alpha_s,
                    "Nefolds": calc.Nefolds
                })

            elif calc.ret == "insuff":
                insuffcount += 1
            elif calc.ret == "noconverge":
                noconvcount += 1
            else:
                errcount += 1

        outfile1.close()
        outfile2.close()

    summary_df = pd.DataFrame(summary_records)
    summary_file = f"{BASE_OUTDIR}/neqs{NEQS}_summary.csv"
    summary_df.to_csv(summary_file, index=False)
    print(f"\nSummary written to {summary_file}")


print('env', sys.executable)

import platform, numpy, scipy, pandas, matplotlib
print(f"Python: {platform.python_version()}")
print(f"CPU Archachitecture: {platform.machine()}")
print(f"NumPy: {numpy.__version__}")
print(f"SciPy: {scipy.__version__}")
print(f"pandas: {pandas.__version__}")
print(f"matplotlib: {matplotlib.__version__}")

%time run_neqs8_models()

Total models: 400
Base λ₅=-8.924610e-09 included: True
Zero included: True
env /Users/epmeador/opt/anaconda3/bin/python
Python: 3.8.5
CPU Architecture: x86_64
NumPy: 1.24.4
SciPy: 1.10.1
pandas: 1.1.3
matplotlib: 3.3.2

=== Running λ5 = -3.0e-08 ===
calcpath runtime: 0.0275 s
  -> nontrivial

=== INTEGRATOR & INITIAL CONDITIONS DIAGNOSTIC ===
Initial values (yinit): [ 5.50000e+00  1.00000e+00  2.09237e-04 -3.42419e-02  2.78972e-04
 -4.60971e-06  6.87065e-08 -3.00000e-08]
  φ0 = 5.500000e+00
  H0 = 1.000000e+00
  ε0 = 2.092370e-04
  σ0 = -3.424190e-02
  λ₂ = 2.789720e-04
  λ₅ = -4.609710e-06
Initial Nefolds = 57.618
GSL integrator class: step_rk4
Expected tolerances: atol=1e-8, rtol=1e-8

  ns = 0.970
  -> Evaluating spectrum 0
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90


1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
spectrum runtime: 54.2208 s
  -> Saving path /Users/epmeador/Desktop/research/rwarthur/inflation_gravitywaves/inflation_code/Slow-Roll Parameters Tests/higgs_potential_tests/neqs8/lam5_-2.9e-08/path_neqs8_lam5-2.9e-08_000.dat

=== Running λ5 = -2.9e-08 ===

125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
spectrum runtime: 51.9264 s
  -> Saving path /Users/epmeador/Desktop/research/rwarthur/inflation_gravitywaves/inflation_code/Slow-Roll Parameters Tests/higgs_potential_tests/neqs8/lam5_-2.9e-08/path_neqs8_lam5-2.9e-08_000.dat

=== Running λ5 = -2.9e-08 ===
calcpath runtime: 0.0254 s
  -> nontrivial

=== INTEGRATOR & INITIAL CONDITIONS DIAGNOSTIC ===
Initial values (yinit): [ 5.50000000e+00  1.00000000e+00  2.09237000e-04 -3.42419000e-02
  2.78972000e-04 -4.60971000e-06  6.87065000e-08 -2.86146096e-08]
  φ0 = 5.500000e+00
  H0 = 1.000000e+00
  ε0 = 2.092370e-04
  σ0 = -3.424190e-02
  λ₂ = 2.789720e-04
  λ₅ = -4.609710e-06
Initial Nefolds

1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
spectrum runtime: 51.5987 s
  -> Saving path /Users/epmeador/Desktop/research/rwarthur/inflation_gravitywaves/inflation_code/Slow-Roll Parameters Tests/higgs_potential_tests/neqs8/lam5_-2.8e-08/path_neqs8_lam5-2.8e-08_000.dat

=== Running λ5 = -2.8e-08 ===

124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
spectrum runtime: 50.7024 s
  -> Saving path /Users/epmeador/Desktop/research/rwarthur/inflation_gravitywaves/inflation_code/Slow-Roll Parameters Tests/higgs_potential_tests/neqs8/lam5_-2.7e-08/path_neqs8_lam5-2.7e-08_000.dat

=== Running λ5 = -2.7e-08 ===
calcpath runtime: 0.0258 s
  -> nontrivial

=== INTEGRATOR & INITIAL CONDITIONS DIAGNOSTIC ===
Initial values (yinit): [ 5.50000000e+00  1.00000000e+00  2.09237000e-04 -3.42419000e-02
  2.78972000e-04 -4.60971000e-06  6.87065000e-08 -2.72292191e-08]
  φ0 = 5.500000e+00
  H0 = 1.000000e+00
  ε0 = 2.092370e-04
  σ0 = -3.424190e-02
  λ₂ = 2.789720e-04
  λ₅ = -4.609710e-06
Initial Nef

...

126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
spectrum runtime: 55.4333 s
  -> Saving path /Users/epmeador/Desktop/research/rwarthur/inflation_gravitywaves/inflation_code/Slow-Roll Parameters Tests/higgs_potential_tests/neqs8/lam5_-1.5e-08/path_neqs8_lam5-1.5e-08_000.dat

=== Running λ5 = -1.5e-08 ===
calcpath runtime: 0.0233 s
  -> nontrivial

=== INTEGRATOR & INITIAL CONDITIONS DIAGNOSTIC ===
Initial values (yinit): [ 5.50000000e+00  1.00000000e+00  2.09237000e-04 -3.42419000e-02
  2.78972000e-04 -4.60971000e-06  6.87065000e-08 -1.48866499e-08]
  φ0 = 5.500000e+00
  H0 = 1.000000e+00
  ε0 = 2.092370e-04
  σ0 = -3.424190e-02
  λ₂ = 2.789720e-04
  λ₅ = -4.609710e-06
Initial Nefolds = 6

1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
spectrum runtime: 54.0155 s
  -> Saving path /Users/epmeador/Desktop/research/rwarthur/inflation_gravitywaves/inflation_code/Slow-Roll Parameters Tests/higgs_potential_tests/neqs8/lam5_-1.4e-08/path_neqs8_lam5-1.4e-08_000.dat

=== Running λ5 = -1.4e-08 ===

126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
spectrum runtime: 52.5828 s
  -> Saving path /Users/epmeador/Desktop/research/rwarthur/inflation_gravitywaves/inflation_code/Slow-Roll Parameters Tests/higgs_potential_tests/neqs8/lam5_-1.4e-08/path_neqs8_lam5-1.4e-08_000.dat

=== Running λ5 = -1.4e-08 ===
calcpath runtime: 0.0253 s
  -> nontrivial

=== INTEGRATOR & INITIAL CONDITIONS DIAGNOSTIC ===
Initial values (yinit): [ 5.50000000e+00  1.00000000e+00  2.09237000e-04 -3.42419000e-02
  2.78972000e-04 -4.60971000e-06  6.87065000e-08 -1.35012594e-08]
  φ0 = 5.500000e+00
  H0 = 1.000000e+00
  ε0 = 2.092370e-04
  σ0 = -3.424190e-02
  λ₂ = 2.789720e-04
  λ₅ = -4.609710e-06
Initial Nefolds = 5

spectrum runtime: 0.1123 s
  -> Saving path /Users/epmeador/Desktop/research/rwarthur/inflation_gravitywaves/inflation_code/Slow-Roll Parameters Tests/higgs_potential_tests/neqs8/lam5_-1.2e-08/path_neqs8_lam5-1.2e-08_000.dat

=== Running λ5 = -1.2e-08 ===
calcpath runtime: 0.0241 s
  -> nontrivial

=== INTEGRATOR & INITIAL CONDITIONS DIAGNOSTIC ===
Initial values (yinit): [ 5.50000000e+00  1.00000000e+00  2.09237000e-04 -3.42419000e-02
  2.78972000e-04 -4.60971000e-06  6.87065000e-08 -1.23677582e-08]
  φ0 = 5.500000e+00
  H0 = 1.000000e+00
  ε0 = 2.092370e-04
  σ0 = -3.424190e-02
  λ₂ = 2.789720e-04
  λ₅ = -4.609710e-06
Initial Nefolds = 63.036
GSL integrator class: step_rk4
Expected tolerances: atol=1e-8, rtol=1e-8

  ns = 0.971
  -> Evaluating spectrum 0
0



KeyboardInterrupt

